Project : Pentaho Log Intelligence

Layer   : Bronze

Notebook: 02_Bronze_Ingestion_Catalina_out

Version : 1.0

Description:
Loads raw Pentaho log files from Unity Catalog Volume

into the Bronze Delta table.

Author: Ernesto Felipe Garay Cervantes


#### Recibimiento de Parametros

In [0]:
import json

dbutils.widgets.text("archivos_nuevos","")

archivos_nuevos = json.loads(dbutils.widgets.get("archivos_nuevos"))

print("====ARCHIVOS RECIBIDOS===")
for archivo in archivos_nuevos:
    print(archivo)




### Configuracion

In [0]:
#configuracion

CATALOG = "pentaho_logs"
SCHEMA = "bronze"
VOLUME_PATH = "/Volumes/pentaho_logs/bronze/volume_catalinalogs"

BRONZE_TABLE_CATALINA = f"{CATALOG}.{SCHEMA}.bronze_logs_catalina"






### Validación Volumen 

In [0]:
#display(dbutils.fs.ls(VOLUME_PATH))



### Lectura de archivos Catalina

In [0]:
from pyspark.sql.functions import col

VOLUME_PATH = "/Volumes/pentaho_logs/bronze/volume_catalinalogs"

archivos_path = [
    f"{VOLUME_PATH}/{archivo}"
    for archivo in archivos_nuevos
]

print("==== PATH PROXIMOS A PROCESARSE ====")

for path in archivos_path:
    print(path)


df_bronze_catalina = (
    spark.read
       .text(archivos_path)
       .select(
           col("value").alias("descripcion"),
        col("_metadata.file_path").alias("file_path"),
        col("_metadata.file_name").alias("file_name")
       )
    )
#display(df_bronze_catalina.limit(10))







### Enriquecimiento DataFrame Catalina

In [0]:

from pyspark.sql.functions import regexp_extract, to_date, current_timestamp,col,when
 
df_bronze_catalina = (df_bronze_catalina.withColumn("application",regexp_extract("file_name", r"^([A-Za-z]+).", 1))
                                        .withColumn("hora",regexp_extract("descripcion", r"(\d{2}:\d{2}:\d{2}\.\d{3})", 1))
                                        .withColumn("fecha",regexp_extract("descripcion",r"(\d{4}-\d{2}-\d{2})", 1))
                                        .withColumn("Nivel",regexp_extract("descripcion",r"\b(INFO|WARNING|SEVERE|DEBUG)\b", 1))
        .withColumn("is_event_start",when(regexp_extract(col("descripcion"), r"\b(INFO|WARNING|SEVERE|DEBUG)\b",0) != "",True).otherwise(False))
         .withColumn("Timestamp",current_timestamp())    

) 

                            

### limpiar data frame 
###df = df.drop("<campo>")
##df_bronze_catalina= df_bronze_catalina.drop("value")
#display(df_bronze_catalina.limit(20))



#### Validaciones DATAFRAME

In [0]:
# Número de registros
print(f"Total de líneas: {df_bronze_catalina.count():,}")

# Estructura
df_bronze_catalina.printSchema()




In [0]:
from pyspark.sql.functions import col, sum, when

df_bronze_catalina.select(
    sum(when(col("descripcion").isNull(), 1).otherwise(0)).alias("descripcion_null"),
    sum(when(col("file_path").isNull(), 1).otherwise(0)).alias("file_path_null"),
    sum(when(col("file_name").isNull(), 1).otherwise(0)).alias("file_name_null"),
    sum(when(col("application").isNull(), 1).otherwise(0)).alias("application_null"),
 sum(when(col("hora").isNull(), 1).otherwise(0)).alias("hora_null"),
 sum(when(col("Nivel").isNull(), 1).otherwise(0)).alias("Nivel_null"),
  sum(when(col("fecha").isNull(), 1).otherwise(0)).alias("fecha_null"),

).show()



### Creación tabla BRONZE_TABLE_CATALINA

In [0]:
BRONZE_TABLE_CATALINA = "pentaho_logs.bronze.bronze_logs_catalina"
(
    df_bronze_catalina.write
        .format("delta")
        .mode("append")
        .saveAsTable(BRONZE_TABLE_CATALINA)
)

#dbutils.notebook.exit("llegue a creación de tabla")

In [0]:
#display(spark.table(BRONZE_TABLE_CATALINA).limit(20))